**Purpose of this Notebook**

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) pipeline using LangChain, OpenAI models, and a FAISS vector database.
The goal is to answer user questions grounded in external documents, rather than relying solely on the language model’s internal knowledge. 

Web Page -> Document Loader -> Text Splitter -> Embeddings -> FAISS Vector Store -> Retriever  -> 
Document Combination (Stuff) -> LLM Answer

**Problem It Solves :** 

Large Language Models cannot read your documents or stay up-to-date. They may hallucinate answers or respond without factual grounding.This notebook solves that by retrieving relevant information from documents first, then asking the model to answer only using that retrieved context.

**How It Is Solved :**  
1) Documents are converted into vector embeddings
2) Relevant content is retrieved using similarity search
3) Retrieved content is injected into the LLM prompt
4) The model generates a context-grounded answer

**Tech Stack & Libraries Used :**  Python , LangChain
- langchain_community – document loaders, FAISS integration
- langchain_openai – embeddings and chat models
- langchain_classic – classic RAG chains
- FAISS – vector similarity search
- OpenAI – embeddings and LLMs
- Jupyter Notebook – interactive development and experimentation

In [ ]:
# Import OS module to interact with environment variables
# This allows us to read and set system-level configuration securely
import os

# Import load_dotenv to read variables from a .env file
# This helps keep API keys and secrets out of source code
from dotenv import load_dotenv

# Load all key-value pairs from the .env file into the process environment
# After this call, values in .env can be accessed using os.getenv()
load_dotenv()


# -----------------------------
# OpenAI Configuration
# -----------------------------

# Set the OpenAI API key explicitly in the environment
# LangChain and OpenAI SDKs automatically look for this variable
# This ensures consistent behavior across notebooks, scripts, and containers
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# -----------------------------
# LangChain / LangSmith Configuration
# -----------------------------

# Set LangChain API key for authentication with LangSmith
# Required for tracing, debugging, and observability of chains and agents
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# Enable LangChain's newer tracing system (Tracking V2)
# This provides better visibility into agents, tool calls, and multi-step chains
# Note: Environment variables must be strings
os.environ["LANGCHAIN_TRACKING_V2"] = "true"

# Assign a project name to group all traces under a single logical experiment
# Useful for separating FAISS tests, RAG pipelines, and production runs
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# ---------------------------------------------
# Step 1: Import required LangChain components
# ---------------------------------------------

# WebBaseLoader:
# Used to fetch and extract readable text content from a web URL
from langchain_community.document_loaders import WebBaseLoader

# OpenAIEmbeddings:
# Converts text chunks into numerical vectors using an OpenAI embedding model
from langchain_openai import OpenAIEmbeddings

# FAISS:
# A high-performance vector store used for similarity search over embeddings
from langchain_community.vectorstores import FAISS

# ChatOpenAI:
# Wrapper around OpenAI chat models for question answering and generation
from langchain_openai import ChatOpenAI


# ---------------------------------------------
# Step 2: Load data from a web page
# ---------------------------------------------

# Initialize a web loader with the target documentation URL
loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/overview")

# Load the web content into LangChain Document objects
# Each Document contains:
#   - page_content: extracted text
#   - metadata: source URL and other context
data = loader.load()


# ---------------------------------------------
# Step 3: Split long text into manageable chunks
# ---------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter:
# Splits text hierarchically (paragraphs → sentences → characters)
# to preserve semantic meaning as much as possible
text_splitters = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Maximum number of characters per chunk
    chunk_overlap=200    # Overlapping characters to preserve context continuity
)

# Split the loaded documents into smaller chunked documents
# These chunks are better suited for embedding and retrieval
doc = text_splitters.split_documents(data)


# ---------------------------------------------
# Step 4: Create embeddings for each text chunk
# ---------------------------------------------

# Initialize the embedding model
# This model maps each text chunk to a dense vector representation
embeddings = OpenAIEmbeddings()


# ---------------------------------------------
# Step 5: Build the FAISS vector database
# ---------------------------------------------

# Create a FAISS vector store from the chunked documents
# Internally:
#   - Each chunk is embedded
#   - Vectors are indexed for fast similarity search
#   - Original text + metadata are stored alongside vectors
vectordb = FAISS.from_documents(doc, embeddings)


# ---------------------------------------------
# Step 6: Initialize the chat-based LLM
# ---------------------------------------------

# Initialize a chat model for answering questions using retrieved context
# This LLM will later be combined with the vector store for RAG
llm = ChatOpenAI(model="gpt-4o")

# Web page → Documents → Chunks → Embeddings → FAISS index → LLM 


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
# Query the vector store db 
query="We recommend you use LangChain if you want to quickly build agents and autonomous applications" 
result=vectordb.similarity_search(query)
result[0].page_content

'See the Installation instructions and Quickstart guide to get started building your own agents and applications with LangChain.\n\u200b Core benefits'

In [ ]:
# ---------------------------------------------------------
# Import legacy "classic" document-combination chain
# ---------------------------------------------------------

# create_stuff_documents_chain:
# A classic LangChain helper that:
#  - Takes multiple retrieved documents
#  - "Stuffs" (concatenates) their text into a single prompt variable {context}
#  - Sends the combined context + user question to the LLM
#
# This is a legacy-style chain preserved in langchain-classic
# for backward compatibility with older RAG patterns
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# ---------------------------------------------------------
# Import prompt template for chat-based LLMs
# ---------------------------------------------------------

# ChatPromptTemplate:
# Used to define structured prompts for chat models
# Supports variable substitution such as {context} and {input}
from langchain_core.prompts import ChatPromptTemplate


# ---------------------------------------------------------
# Define the prompt used for RAG answering
# ---------------------------------------------------------

# This prompt instructs the LLM to:
#  - Answer strictly using the retrieved document context
#  - Avoid using outside knowledge
#  - Use {context} for retrieved documents
#  - Use {input} for the user's question
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}
"""
)

# ---------------------------------------------------------
# Create the document-combination (stuff) chain
# ---------------------------------------------------------

# create_stuff_documents_chain:
#  - Takes an LLM and a prompt
#  - Expects a list of Document objects as input
#  - Automatically formats the documents into {context}
#  - Produces a runnable chain that outputs an LLM response
#
# This chain is typically used after a retriever step in RAG
document_chain = create_stuff_documents_chain(llm, prompt)


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template=' \nAnswer the following question based only based on the provided context:\n<context> \n{context}\n</context>\n\nQuestion: {input} \n'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, 

In [ ]:
# ---------------------------------------------------------
# Step 1: Convert the vector database into a retriever
# ---------------------------------------------------------

# as_retriever():
# Wraps the FAISS vector store with a standard retriever interface
# The retriever:
#  - Takes a user query (string)
#  - Converts it to an embedding
#  - Finds the most similar document chunks from FAISS
#  - Returns a list of Document objects
retriever = vectordb.as_retriever()


# ---------------------------------------------------------
# Step 2: Import the classic retrieval chain helper
# ---------------------------------------------------------

# create_retrieval_chain:
# A classic LangChain helper that connects:
#  - A retriever (FAISS, Chroma, etc.)
#  - A document-combination chain (e.g., stuff documents chain)
#
# This function orchestrates the full RAG flow:
# Question → Retrieve documents → Combine documents → LLM → Answer
from langchain_classic.chains import create_retrieval_chain


# ---------------------------------------------------------
# Step 3: Create the full Retrieval-Augmented Generation chain
# ---------------------------------------------------------

# create_retrieval_chain(retriever, document_chain):
#  - Takes the user input (key: "input")
#  - Uses the retriever to fetch relevant documents
#  - Passes those documents to document_chain
#  - document_chain "stuffs" documents into {context}
#  - Calls the LLM using the provided prompt
#  - Returns a structured result containing the answer and context
retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [17]:
response=retrieval_chain.invoke({"input":"We recommend you use LangChain if you want to quickly build agents and autonomous applications"})
response['answer']

'True. LangChain is recommended if you want to quickly build agents and autonomous applications, as it provides pre-built agent architecture and model integrations to help you get started quickly.'

In [3]:
!pip install -U "crewai[rag]"

  Using cached crewai-1.8.1-py3-none-any.whl.metadata (36 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/710.0 kB ? eta -:--:--
   -------------- ------------------------- 262.1/710.0 kB ? eta -:--:--
   ---------------------------------------- 710.0/710.0 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 8.5 MB/s eta 0:00:03
   ---- ----------------------------------- 2.4/19.8 MB 6.1 MB/s eta 0:00:03
   -------- ------------------------------- 4.5/19.8 MB 7.7 MB/s eta 0:00:03
   ------------ -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 1.1.1 which is incompatible.
langchain-openai 1.1.6 requires openai<3.0.0,>=1.109.1, but you have openai 1.83.0 which is incompatible.
transformers 4.57.3 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.
